In [4]:
import numpy as np 
import pandas as pd
import casadi as ca

# Optimisation

**Anna Bergougnoux Santini et Julien colella**  
Avril 2026  

---

## Étude du problème d'optimisation

1. Le revenu d'une entreprise s'écrit :  

   $$
   R(y)= p(y)y - c(y)
   $$

   Ici, la formule va de soi : on multiplie la matrice des quantités de matière première commandées $c=(c_i)$ avec la matrice coût unitaire $r=(r_i)$.

   $min(q,d)$ correspond à la quantité vendue :  
   - si $q > d$, comme on ne peut pas vendre plus que la demande, la quantité vendue est donc $d$  
   - si $q < d$, on vend une quantité $q$

2. Ce terme-ci n'est pas différentiable.

3. Soit $\alpha \gg 1$ :

4. On cherche alors à minimiser :

   $$
   g(q,d) = - v^T h(q, d) + c^T r
   $$

   sous la contrainte :

   $$
   Aq - r \le 0
   $$

   On peut donc réécrire ceci avec $z=(q,r)$ :

   $$
   g(z) = - v^T h(z_1, d) + c^T z_2
   $$

5. Ainsi que :

   $$
   c(z) = A z_1 - z_2
   $$

   **ATTENTION : il manque les contraintes de positivité**

   $z$ dépend donc des deux vecteurs $q$ et $r$, et donc des $q_j$ et des $r_i$.

---

## Étude et résolution numérique

6. Pour ce problème, qui est donc un problème d'optimisation sous contrainte, où $f$ est différentiable non-linéaire et $c$ est une contrainte linéaire, on peut penser aux conditions de KKT.



**Question 6**

In [ ]:
alpha = 0.1
c = 1e-3 * np.array([30,1,1.3,4,1])
v = np.array([0.9,1.5,1.1])
d = np.array([400,67,33])
A = np.array([
    [3.5,2,1],
    [250,80,25],
    [0,8,3],
    [0,40,10],
    [0,8.5,0]
])

m = 5
p = 3

c = ca.DM(c)
v = ca.DM(v)
#on a eu un problème avec les multiplications entre array et objets casadi, 
#on a donc converti les array de facon à ce que les ojets manipulés soient compatibles

r = ca.MX.sym('r', m)
q = ca.MX.sym('q', p)


def h(q, d, alpha):
    return (q*ca.exp(-alpha*q) + d*ca.exp(-alpha*d)) / (ca.exp(-alpha*q) + ca.exp(-alpha*d))

f = c.T @ r - v.T @ h(q, d, alpha)


#contraintes
c1 = A @ q - r
c2 = -r
c3 = -q

c = ca.vertcat(c1, c2, c3)


nlp = {'x': ca.vertcat(r,q), 'f': f, 'g': c}

solver = ca.nlpsol('solver', 'ipopt', nlp)


sol = solver(lbg=-ca.inf, ubg=0)

z_opt = sol['x']
print(z_opt)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:       28
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        8
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality c

**Question 7**
